# Evaluate TD3 on Fixed Obstacles

Load the saved TD3 checkpoint and evaluate it greedily on a fixed unseen-obstacle layout, then on randomised layouts for a generalisation check.

## Imports and Paths

In [9]:
from pathlib import Path
import sys

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
    if BASE_DIR.name != "Continuous_Diff_Drive":
        BASE_DIR = BASE_DIR / "Continuous_Diff_Drive"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from diff_drive_agent import DiffDriveTD3Agent
from diff_drive_env import DiffDriveEnv

BASE_DIR

PosixPath('/Users/alessiotimofte/PycharmProjects/NAML_project/Continuous_Diff_Drive')

## Fixed Evaluation Configuration

In [10]:
CHECKPOINT_PATH        = BASE_DIR / "models" / "td3_checkpoint.pt"
EVALUATION_VIDEO_DIR   = BASE_DIR / "videos" / "evaluation_td3"
EVALUATION_NAME_PREFIX = "td3_diff_drive_eval_unseen_fixed_obstacles_greedy"

# Fixed obstacle layout (unseen during training)
OBSTACLES = [
    (2.0, 4.0, 3.0, 0.3),
    (5.0, 2.0, 0.3, 3.0),
    (7.0, 6.0, 1.5, 0.3),
]

ENV_KWARGS = dict(
    room_size       = (10.0, 10.0),
    obstacles       = OBSTACLES,
    random_obst     = False,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
)

ENV_KWARGS_RANDOM = dict(
    room_size       = (10.0, 10.0),
    obstacles       = None,
    random_obst     = True,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
)

# Hyperparameters MUST match those used during training so the agent
# constructs identical-shaped networks (otherwise load_state_dict fails)
ACTOR_LR          = 1e-4
CRITIC_LR         = 1e-3
DISCOUNT          = 0.99
TAU               = 0.005
POLICY_DELAY      = 2
TARGET_NOISE_STD  = 0.2
TARGET_NOISE_CLIP = 0.5
EXPL_NOISE_STD    = 0.2     # matches training (raw action units, DDPG-style)
EXPL_NOISE_CLIP   = 0.5     # matches training
BATCH_SIZE        = 256
BUFFER_SIZE       = 300_000
HIDDEN_DIM        = 256
WARMUP_STEPS      = 5_000
DEVICE            = "cpu"
N_EPISODES        = 3
N_RANDOM_EPISODES = 10

EVALUATION_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH

PosixPath('/Users/alessiotimofte/PycharmProjects/NAML_project/Continuous_Diff_Drive/models/td3_checkpoint.pt')

## Environment and Agent

In [11]:
env        = DiffDriveEnv(**ENV_KWARGS)
env_random = DiffDriveEnv(**ENV_KWARGS_RANDOM)

AGENT_KWARGS = dict(
    actor_lr          = ACTOR_LR,
    critic_lr         = CRITIC_LR,
    discount          = DISCOUNT,
    tau               = TAU,
    policy_delay      = POLICY_DELAY,
    target_noise_std  = TARGET_NOISE_STD,
    target_noise_clip = TARGET_NOISE_CLIP,
    expl_noise_std    = EXPL_NOISE_STD,
    expl_noise_clip   = EXPL_NOISE_CLIP,
    batch_size        = BATCH_SIZE,
    buffer_size       = BUFFER_SIZE,
    hidden_dim        = HIDDEN_DIM,
    warmup_steps      = WARMUP_STEPS,
    device            = DEVICE,
)

agent        = DiffDriveTD3Agent(env=env,        **AGENT_KWARGS)
agent_random = DiffDriveTD3Agent(env=env_random, **AGENT_KWARGS)

## Load Checkpoint

In [12]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No checkpoint found at {CHECKPOINT_PATH}")

agent.load_checkpoint(CHECKPOINT_PATH, load_optimizers=False)
agent_random.load_checkpoint(CHECKPOINT_PATH, load_optimizers=False)

TD3 checkpoint loaded from /Users/alessiotimofte/PycharmProjects/NAML_project/Continuous_Diff_Drive/models/td3_checkpoint.pt
TD3 checkpoint loaded from /Users/alessiotimofte/PycharmProjects/NAML_project/Continuous_Diff_Drive/models/td3_checkpoint.pt


{'actor': OrderedDict([('action_scale', tensor([0.7500, 3.1416])),
              ('action_bias', tensor([0.2500, 0.0000])),
              ('net.0.weight',
               tensor([[ 9.1330e-02,  2.2194e-01,  1.7473e-01,  ...,  2.9271e-01,
                        -4.0649e-01, -5.2072e-01],
                       [ 1.6086e-01,  3.1976e-01,  4.5212e-02,  ..., -2.0443e-01,
                        -1.2524e-01,  5.6482e-01],
                       [ 8.7940e-02,  4.7605e-02,  2.6873e-01,  ...,  2.0221e-01,
                        -9.5311e-02,  3.8373e-01],
                       ...,
                       [ 3.6097e-03, -4.6989e-03, -7.2203e-03,  ..., -3.9960e-03,
                         5.5102e-03,  4.7160e-04],
                       [-5.8096e-03, -3.7981e-04, -1.3587e-03,  ...,  3.8747e-03,
                        -2.3577e-03, -3.7805e-03],
                       [-1.1898e-01, -4.6579e-02,  1.0286e-02,  ..., -2.7850e-01,
                        -2.9419e-02, -7.0025e-02]])),
              ('

## Greedy Evaluation on Fixed Obstacles

In [13]:
agent.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = EVALUATION_NAME_PREFIX,
    n_episodes   = N_EPISODES,
)

  TD3 eval ep 1: reward = 165.0 | steps = 133 | goal reached: yes
  TD3 eval ep 2: reward = 165.0 | steps = 133 | goal reached: yes
  TD3 eval ep 3: reward = 165.0 | steps = 133 | goal reached: yes


## Evaluation Videos (Fixed)

In [14]:
from IPython.display import Video, display

video_paths = sorted(EVALUATION_VIDEO_DIR.glob(f"{EVALUATION_NAME_PREFIX}*.mp4"))
if not video_paths:
    print(f"No videos found yet in {EVALUATION_VIDEO_DIR}")
for video_path in video_paths:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))

td3_diff_drive_eval_unseen_fixed_obstacles_greedy-episode-0.mp4


td3_diff_drive_eval_unseen_fixed_obstacles_greedy-episode-1.mp4


td3_diff_drive_eval_unseen_fixed_obstacles_greedy-episode-2.mp4


## Random Obstacle Evaluation (generalisation)

In [15]:
RANDOM_NAME_PREFIX = "td3_diff_drive_eval_random_obstacles_greedy"
agent_random.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = RANDOM_NAME_PREFIX,
    n_episodes   = N_RANDOM_EPISODES,
)

  TD3 eval ep 1: reward = -104.6 | steps = 121 | goal reached: no
  TD3 eval ep 2: reward = 174.9 | steps = 108 | goal reached: yes
  TD3 eval ep 3: reward = 174.4 | steps = 109 | goal reached: yes
  TD3 eval ep 4: reward = 175.3 | steps = 109 | goal reached: yes
  TD3 eval ep 5: reward = 164.5 | steps = 135 | goal reached: yes
  TD3 eval ep 6: reward = 165.1 | steps = 131 | goal reached: yes
  TD3 eval ep 7: reward = 176.2 | steps = 107 | goal reached: yes
  TD3 eval ep 8: reward = 159.0 | steps = 138 | goal reached: yes
  TD3 eval ep 9: reward = 162.4 | steps = 136 | goal reached: yes
  TD3 eval ep 10: reward = 175.4 | steps = 107 | goal reached: yes


In [16]:
video_paths = sorted(EVALUATION_VIDEO_DIR.glob(f"{RANDOM_NAME_PREFIX}*.mp4"))
if not video_paths:
    print(f"No videos found yet in {EVALUATION_VIDEO_DIR}")
for video_path in video_paths:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))

td3_diff_drive_eval_random_obstacles_greedy-episode-0.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-1.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-2.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-3.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-4.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-5.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-6.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-7.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-8.mp4


td3_diff_drive_eval_random_obstacles_greedy-episode-9.mp4
